# Relatório 2: CNC
Sistemas de equações lineares e não lineares: Fatoração LU, Cholesky, Gauss-Jacobi, Gauss-Seidel, Newton e Newton Modificado.

## Imports

In [2]:
import numpy as np
import sympy as sp

## Biblioteca de métodos
As funções são as mesmas dos arquivos em `methods/`. Duas adaptações pro que o enunciado pede: Jacobi/Seidel recebem `x0` e usam o critério duplo, e Newton/Newton Modificado recebem `F` e `J` como funções (jacobiana analítica) e resolvem `J Δx = -F` com a nossa LU.

### Impressão de tabelas

In [3]:
# imprime uma tabela simples: colunas = lista de nomes, linhas = lista de listas
def imprimir_tabela(colunas, linhas):
    print(" | ".join(f"{c:>12}" for c in colunas))
    print("-" * (15 * len(colunas)))
    for linha in linhas:
        print(" | ".join(f"{'-':>12}" if v is None else f"{v:>12.6g}" for v in linha))

### Fatoração LU com pivotação parcial (fig. 2.7)
`lu` decompõe, `extrair_lu` separa L e U, `resolver_lu` resolve `L y = P b` e `U x = y`.

In [4]:
def lu(A):
    # copia pra n estragar a matriz original (o algoritmo escreve dentro da própria A)
    A = np.array(A, dtype=float)
    m, n = A.shape

    pivot = np.arange(m) # começa [0, 1, 2, ..., m-1] (sem nenhuma troca)
    pdu = 1.0
    info = 0

    for j in range(min(m, n)):
        # escolha do pivô: procura na coluna j (da diagonal pra baixo) o maior elemento em módulo
        p = j
        amax = abs(A[j, j])
        for k in range(j + 1, m):
            if abs(A[k, j]) > amax:
                amax = abs(A[k, j])
                p = k

        # se o maior não tá na diagonal, troca a linha j com a linha p (a linha inteira, incluindo os multiplicadores já guardados)
        if p != j:
            for k in range(n):
                A[j, k], A[p, k] = A[p, k], A[j, k]
            pivot[j], pivot[p] = pivot[p], pivot[j] # registra a troca
            pdu = -pdu # cada troca de linha inverte o sinal do determinante

        pdu = pdu * A[j, j]

        # eliminação de Gauss
        if abs(A[j, j]) != 0:
            r = 1 / A[j, j]
            for i in range(j + 1, m):
                mult = A[i, j] * r # multiplicador m_ij
                A[i, j] = mult     # guarda ele no lugar do zero (essa parte é o L)
                for k in range(j + 1, n):
                    A[i, k] = A[i, k] - mult * A[j, k]
        else:
            # pivô zero = matriz singular. Guarda só o primeiro que aparecer
            if info == 0:
                info = j + 1

    # A agora é U + L - I (U na diagonal e acima, multiplicadores do L abaixo da diagonal)
    return A, pivot, pdu, info

# separa a matriz decomposta em L e U, só pra poder ver/conferir
def extrair_lu(A_lu):
    m, n = A_lu.shape
    k = min(m, n)
    L = np.tril(A_lu[:, :k], -1) + np.eye(m, k) # abaixo da diagonal + 1s na diagonal
    U = np.triu(A_lu[:k, :])                    # diagonal e acima
    return L, U

# resolve Ax = b usando a decomposição (só funciona pra A quadrada e não singular)
def resolver_lu(A_lu, pivot, b):
    n = A_lu.shape[0]
    pb = np.array(b, dtype=float)[pivot] # P b: aplica as mesmas trocas de linha no b

    # L y = P b  (substituição progressiva, de cima pra baixo)
    y = np.zeros(n)
    for i in range(n):
        y[i] = pb[i] - np.dot(A_lu[i, :i], y[:i]) # a diagonal de L é 1, n precisa dividir

    # U x = y  (substituição retroativa, de baixo pra cima)
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - np.dot(A_lu[i, i+1:], x[i+1:])) / A_lu[i, i]

    return x

### Fatoração de Cholesky (fig. 2.10)
`A = G Gᵀ`, só pra matriz simétrica e definida positiva. Se o radicando ficar ≤ 0, retorna `info = k`.

In [5]:
def cholesky(A):
    # copia pra n estragar a matriz original (o algoritmo escreve dentro da própria A)
    A = np.array(A, dtype=float)
    n = A.shape[0]

    det = 1.0
    info = 0

    # o cálculo avança coluna a coluna: 1o o elemento da diagonal, depois os que estão abaixo dele
    for j in range(n):
        # elemento da diagonal: g_jj = raiz(a_jj - soma dos g_jk²)
        soma = 0.0
        for k in range(j):
            soma = soma + A[j, k] * A[j, k]
        t = A[j, j] - soma # o que vai dentro da raiz

        if t > 0:
            A[j, j] = np.sqrt(t)
            r = 1 / A[j, j]
            det = det * t # t = g_jj², o produto de todos eles é o det(A)
        else:
            # radicando <= 0 em algum passo = A n é definida positiva. O próprio algoritmo é o teste
            info = j + 1
            print("matriz não é definida positiva")
            return A, det, info

        # elementos abaixo da diagonal (i > j): g_ij = (a_ij - soma de g_ik * g_jk) / g_jj
        for i in range(j + 1, n):
            soma = 0.0
            for k in range(j):
                soma = soma + A[i, k] * A[j, k]
            A[i, j] = (A[i, j] - soma) * r # multiplica por 1/g_jj (mesma coisa que dividir)

    # a G tá no triângulo inferior de A. O de cima ainda tem os valores originais, por isso o extrair_g
    return A, det, info

# pega só o triângulo inferior (diagonal inclusa) da matriz decomposta, pq acima da diagonal g_ij = 0
def extrair_g(A_g):
    return np.tril(A_g)

# resolve Ax = b usando a decomposição
def resolver_cholesky(A_g, b):
    G = extrair_g(A_g)
    n = G.shape[0]
    b = np.array(b, dtype=float)

    # G y = b  (substituição progressiva, de cima pra baixo)
    y = np.zeros(n)
    for i in range(n):
        y[i] = (b[i] - np.dot(G[i, :i], y[:i])) / G[i, i]

    # G^T x = y  (substituição retroativa, de baixo pra cima)
    GT = G.T
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - np.dot(GT[i, i+1:], x[i+1:])) / GT[i, i]

    return x

### Gauss-Jacobi e Gauss-Seidel (fig. 2.11 e 2.12)
Com `x0` como parâmetro e parada quando **os dois** critérios valem juntos: `||x(k+1) - x(k)||∞ / max(||x(k+1)||∞, 1) < toler_rel` e `||b - A x(k+1)||∞ < toler_res`.

In [6]:
# toler_rel: tolerância do erro relativo entre iterações
# toler_res: tolerância do resíduo ||b - A x||inf
def jacobi(A, b, x0, toler_rel, toler_res, iter_max):
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    n = len(b)
    x = np.array(x0, dtype=float)

    k = 0
    historico = [] # [k, x1..xn, dr, resíduo]

    while True:
        k = k + 1

        # v é o vetor novo. A gente n mexe no x enquanto calcula, senão virava Seidel
        v = np.zeros(n)
        for i in range(n):
            soma = 0.0
            for j in range(n):
                if i != j:
                    soma = soma + A[i, j] * x[j]
            v[i] = (b[i] - soma) / A[i, i]

        dr = np.max(np.abs(v - x)) / max(np.max(np.abs(v)), 1)
        x = v
        res = np.max(np.abs(b - A @ x))
        historico.append([k, *x, dr, res])

        # os 2 critérios têm que valer juntos
        if (dr < toler_rel and res < toler_res) or k >= iter_max:
            break

    info = 0 if (dr < toler_rel and res < toler_res) else 1
    return x, k, info, historico

def seidel(A, b, x0, toler_rel, toler_res, iter_max):
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    n = len(b)
    x = np.array(x0, dtype=float)

    k = 0
    historico = []

    while True:
        k = k + 1
        x_ant = x.copy() # guarda o x(k) pra calcular a diferença depois

        for i in range(n):
            # aqui o x já tem os valores novos pra j < i e os antigos pra j > i
            soma = 0.0
            for j in range(n):
                if i != j:
                    soma = soma + A[i, j] * x[j]
            x[i] = (b[i] - soma) / A[i, i] # atualiza na hora

        dr = np.max(np.abs(x - x_ant)) / max(np.max(np.abs(x)), 1)
        res = np.max(np.abs(b - A @ x))
        historico.append([k, *x, dr, res])

        if (dr < toler_rel and res < toler_res) or k >= iter_max:
            break

    info = 0 if (dr < toler_rel and res < toler_res) else 1
    return x, k, info, historico

### Newton (ruggiero, cap. 4)
`J(x_k) Δx = -F(x_k)` a cada iteração, reaproveitando `lu`/`resolver_lu`. Critério adaptado do enunciado do exercício 3: `||Δx||∞ / max(||x(k+1)||∞, 1) < toler_rel` **ou** `||F(x(k+1))||∞ < toler_res`.

In [7]:
def newton(F, J, x0, toler_rel, toler_res, k_max):
    x = np.array(x0, dtype=float)
    k = 0
    info = 0
    historico = [] # [k, x1..xn, Δx1..Δxn, dr, ||F||]
    # k = 0 só mostra o ponto inicial
    historico.append([0, *x, *[None]*len(x), None, np.max(np.abs(F(x)))])

    while True:
        # monta J(x_k) e F(x_k), fatora J e resolve J Δx = -F  (letra c)
        J_lu, pivot, pdu, info_lu = lu(J(x))
        if info_lu != 0:
            print("jacobiana singular, escolha outro x0")
            info = 2
            break
        dx = resolver_lu(J_lu, pivot, -F(x))

        x = x + dx
        k = k + 1
        dr = np.max(np.abs(dx)) / max(np.max(np.abs(x)), 1)
        norma_f = np.max(np.abs(F(x)))
        historico.append([k, *x, *dx, dr, norma_f])

        # os critérios aqui são alternativos (OU)
        if dr < toler_rel or norma_f < toler_res:
            break
        if k >= k_max:
            info = 1
            break

    return x, k, info, historico

### Newton Modificado
A jacobiana é calculada e fatorada **uma vez**, em `x(0)`, e reaproveitada em todas as iterações: cada iteração custa só duas substituições. Convergência linear em vez de quadrática.

In [8]:
def newton_modificado(F, J, x0, toler_rel, toler_res, k_max):
    x = np.array(x0, dtype=float)

    # a diferença pro newton: calcula J(x0) e fatora LU só UMA vez, antes do loop
    J_lu, pivot, pdu, info_lu = lu(J(x))
    if info_lu != 0:
        print("jacobiana singular em x0, escolha outro x0")
        return x, 0, 2, []

    k = 0
    info = 0
    historico = [[0, *x, None, None, np.max(np.abs(F(x)))]] # [k, x1..xn, dx, dr, ||F||]

    while True:
        # resolve J(x0) Δx = -F(x_k) reaproveitando a LU (só 2 substituições)
        dx = resolver_lu(J_lu, pivot, -F(x))
        x = x + dx
        k = k + 1
        norma_dx = np.max(np.abs(dx))
        dr = norma_dx / max(np.max(np.abs(x)), 1)
        norma_f = np.max(np.abs(F(x)))
        historico.append([k, *x, norma_dx, dr, norma_f])

        if dr < toler_rel or norma_f < toler_res:
            break
        if k >= k_max:
            info = 1
            break

    return x, k, info, historico

## Exercício 1: Transferência de calor (LU e Cholesky)
Sistema `A T = b` com 4 temperaturas nodais (°C). Letras: (a) simetria, (b) dominância diagonal, (c) autovalores e definida positiva, (d) LU, (e) Cholesky, (f) resíduo `||b - A x||∞` das duas soluções, (g) efeito de trocar `b` por `b_novo`.

### Dados

In [9]:
A = np.array([[ 7, -2, -1,  0],
              [-2,  6, -1, -1],
              [-1, -1,  7, -2],
              [ 0, -1, -2,  7]])
b = np.array([400, 40, 240, 80])
b_novo = np.array([480, 40, 240, 80])

### a) simetria

In [10]:
# A é simétrica se A = A^T, ou seja a_ij = a_ji pra todo i, j
print("a) simétrica?", np.array_equal(A, A.T), "| max|A - A^T| =", np.max(np.abs(A - A.T)))

a) simétrica? True | max|A - A^T| = 0


**Resposta da letra a:** sim, A = A^T (a diferença é exatamente 0). Cholesky não é descartado por simetria.

### b) dominância diagonal

In [11]:
# por linhas:  |a_ii| > soma dos |a_ij|, j != i
# por colunas: mesma coisa olhando pras colunas
print("\nb) dominância diagonal")
for i in range(4):
    soma_linha = np.sum(np.abs(A[i, :])) - abs(A[i, i])
    soma_coluna = np.sum(np.abs(A[:, i])) - abs(A[i, i])
    print(f"  linha {i+1}: {abs(A[i,i])} > {soma_linha} ? {abs(A[i,i]) > soma_linha} | coluna {i+1}: {abs(A[i,i])} > {soma_coluna} ? {abs(A[i,i]) > soma_coluna}")


b) dominância diagonal
  linha 1: 7 > 3 ? True | coluna 1: 7 > 3 ? True
  linha 2: 6 > 4 ? True | coluna 2: 6 > 4 ? True
  linha 3: 7 > 4 ? True | coluna 3: 7 > 4 ? True
  linha 4: 7 > 3 ? True | coluna 4: 7 > 3 ? True


**Resposta da letra b:** A é estritamente diagonalmente dominante por linhas e por colunas (as 4 linhas e as 4 colunas passam). Isso garante que A é não singular e que Jacobi/Seidel convergiriam. Sendo simétrica com diagonal positiva e dominante, também é definida positiva.

### c) autovalores / definida positiva

In [12]:
# A é simétrica, então uso eigvalsh (própria pra simétricas, devolve autovalores reais em ordem crescente)
autovalores = np.linalg.eigvalsh(A)
print("\nc) autovalores =", autovalores)
print("   todos > 0 ?", np.all(autovalores > 0))

# confirmação pelo critério de sylvester: todos os menores principais com det > 0
menores = [float(np.linalg.det(A[:k, :k])) for k in range(1, 5)]
print("   menores principais =", [round(m, 6) for m in menores])
print("   cond2(A) = λmax/λmin =", autovalores[-1] / autovalores[0])


c) autovalores = [3.11985506 6.22683301 8.24642152 9.40689042]
   todos > 0 ? True
   menores principais = [7.0, 38.0, 249.0, 1507.0]
   cond2(A) = λmax/λmin = 3.0151690463031593


**Resposta da letra c:** autovalores ≈ 3,12 ; 6,23 ; 8,25 ; 9,41, todos positivos, e os menores principais (7, 38, 249, 1507) também. Logo A é simétrica e definida positiva: LU sem pivotação existe e é única, e Cholesky pode ser aplicado. O menor autovalor (3,12) longe de zero e cond2(A) = λmax/λmin ≈ 3 indicam que A é bem condicionada.

### d) LU

In [13]:
A_lu, pivot, pdu, info = lu(A)
L, U = extrair_lu(A_lu)
T_lu = resolver_lu(A_lu, pivot, b)
print("\nd) LU")
print("L =\n", L)
print("U =\n", U)
print("pivot =", pivot, "| det(A) =", pdu, "| info =", info)
print("T (LU) =", T_lu)


d) LU
L =
 [[ 1.          0.          0.          0.        ]
 [-0.28571429  1.          0.          0.        ]
 [-0.14285714 -0.23684211  1.          0.        ]
 [ 0.         -0.18421053 -0.34136546  1.        ]]
U =
 [[ 7.         -2.         -1.          0.        ]
 [ 0.          5.42857143 -1.28571429 -1.        ]
 [ 0.          0.          6.55263158 -2.23684211]
 [ 0.          0.          0.          6.05220884]]
pivot = [0 1 2 3] | det(A) = 1507.0 | info = 0
T (LU) = [80.55739881 50.2189781  63.46383543 36.73523557]


### e) Cholesky

In [14]:
A_g, det_ch, info_ch = cholesky(A)
G = extrair_g(A_g)
T_ch = resolver_cholesky(A_g, b)
print("\ne) Cholesky")
print("G =\n", G)
print("det(A) =", det_ch, "| info =", info_ch)
print("T (Cholesky) =", T_ch)
# os dois determinantes (LU e Cholesky) batem, e o menor principal de ordem 4 acima também (1507)


e) Cholesky
G =
 [[ 2.64575131  0.          0.          0.        ]
 [-0.75592895  2.32992949  0.          0.        ]
 [-0.37796447 -0.55182541  2.55981085  0.        ]
 [ 0.         -0.42919754 -0.87383101  2.46012374]]
det(A) = 1507.0 | info = 0
T (Cholesky) = [80.55739881 50.2189781  63.46383543 36.73523557]


### f) resíduo

In [15]:
res_lu = np.max(np.abs(b - A @ T_lu))
res_ch = np.max(np.abs(b - A @ T_ch))
print("\nf) ||b - A x||inf  LU =", res_lu, "| Cholesky =", res_ch)
print("   diferença entre as soluções =", np.max(np.abs(T_lu - T_ch)))


f) ||b - A x||inf  LU = 5.684341886080802e-14 | Cholesky = 5.684341886080802e-14
   diferença entre as soluções = 7.105427357601002e-15


**Resposta da letra f:** os dois resíduos ficam na ordem de 1e-14 (erro de arredondamento do ponto flutuante), ou seja, as duas soluções satisfazem o sistema até a precisão da máquina e são praticamente idênticas.

### g) b novo

In [16]:
# aqui a gente aproveita a fatoração que já tá feita: só 2 substituições, n precisa fatorar de novo (esse é o ponto da aula 8)
T_novo = resolver_lu(A_lu, pivot, b_novo)
T_novo_ch = resolver_cholesky(A_g, b_novo)
delta = T_novo - T_lu
print("\ng) b novo")
print("T  =", T_lu)
print("T' =", T_novo, "(LU) |", T_novo_ch, "(Cholesky)")
print("ΔT =", delta)
print("ΔT em % =", 100 * delta / T_lu)


g) b novo
T  = [80.55739881 50.2189781  63.46383543 36.73523557]
T' = [93.93497014 55.47445255 66.59588587 38.38088918] (LU) | [93.93497014 55.47445255 66.59588587 38.38088918] (Cholesky)
ΔT = [13.37757133  5.25547445  3.13205043  1.64565362]
ΔT em % = [16.6062603  10.46511628  4.93517357  4.47976879]


**Resposta da letra g:** b1 subiu 80 (de 400 pra 480, +20%). As temperaturas sobem todas, e o efeito diminui com a "distância" do nó 1:

- T1 sobe ≈ 13,4 °C (+16,6%), T2 ≈ 5,3 °C, T3 ≈ 3,1 °C e T4 ≈ 1,6 °C.

Fisicamente b_i é a fonte de calor (ou condição de contorno) do nó i. Aquecer o nó 1 esquenta mais ele e os vizinhos diretos (T2 e T3, que têm a_12 e a_13 ≠ 0). O nó 4 n tem ligação direta com o 1 (a_14 = 0), então só sente o aumento "de segunda mão", via nós 2 e 3, e muito menos. O sistema é linear, então ΔT = A^-1 Δb: dobrar a fonte dobra o efeito. Todos os ΔT são positivos pq a inversa dessa A só tem entradas positivas (aumentar a fonte nunca esfria nenhum nó).

## Exercício 2: Rede de correntes (Gauss-Jacobi e Gauss-Seidel)
Sistema `A V = b` com 5 tensões nodais, `x(0) = 0`. Parada simultânea: `||x(k+1) - x(k)||∞ / max(||x(k+1)||∞, 1) < 1e-6` e `||b - A x(k+1)||∞ < 1e-8`.

### Dados

In [17]:
A = np.array([[10, -2, -1,  0,  0],
              [-2,  9, -2, -1,  0],
              [-1, -2, 10, -2, -1],
              [ 0, -1, -2,  9, -2],
              [ 0,  0, -1, -2,  8]])
b = np.array([12, 5, 0, 5, 10])
x0 = np.zeros(5)
n = 5

### a) verificações prévias

In [18]:
print("a) dominância diagonal")
for i in range(n):
    soma_linha = np.sum(np.abs(A[i, :])) - abs(A[i, i])
    soma_coluna = np.sum(np.abs(A[:, i])) - abs(A[i, i])
    print(f"  linha {i+1}: {abs(A[i,i])} > {soma_linha} ? {abs(A[i,i]) > soma_linha} | coluna {i+1}: {abs(A[i,i])} > {soma_coluna} ? {abs(A[i,i]) > soma_coluna}")

# critério de sassenfeld (só vale pro seidel): beta_i = (soma_{j<i} |a_ij| beta_j + soma_{j>i} |a_ij|) / |a_ii|
beta = np.zeros(n)
for i in range(n):
    s = 0.0
    for j in range(n):
        if j < i:
            s = s + abs(A[i, j]) * beta[j]
        elif j > i:
            s = s + abs(A[i, j])
    beta[i] = s / abs(A[i, i])
print("  sassenfeld: beta =", np.round(beta, 4), "| max =", round(float(np.max(beta)), 4))

# raio espectral das matrizes de iteração (condição necessária e suficiente). M_jacobi = -D^-1 (L+U), M_seidel = -(D+L)^-1 U
D = np.diag(np.diag(A.astype(float)))
M_j = -np.linalg.inv(D) @ (A - D)
M_s = -np.linalg.inv(np.tril(A.astype(float))) @ np.triu(A.astype(float), 1)
print("  rho(M_jacobi) =", max(abs(np.linalg.eigvals(M_j))), "| rho(M_seidel) =", max(abs(np.linalg.eigvals(M_s))))

a) dominância diagonal
  linha 1: 10 > 3 ? True | coluna 1: 10 > 3 ? True
  linha 2: 9 > 5 ? True | coluna 2: 9 > 5 ? True
  linha 3: 10 > 6 ? True | coluna 3: 10 > 6 ? True
  linha 4: 9 > 5 ? True | coluna 4: 9 > 5 ? True
  linha 5: 8 > 3 ? True | coluna 5: 8 > 3 ? True
  sassenfeld: beta = [0.3    0.4    0.41   0.3578 0.1407] | max = 0.41
  rho(M_jacobi) = 0.5022908201991552 | rho(M_seidel) = 0.264362674864085


**Resposta da letra a:** A é estritamente diagonalmente dominante por linhas e por colunas (10>3, 9>5, 10>6, 9>5, 8>3 nas linhas). Sassenfeld: max beta ≈ 0,41 < 1. Raio espectral: ρ(M_J) ≈ 0,50 < 1 e ρ(M_S) ≈ 0,26 < 1. Os 3 testes garantem convergência dos dois métodos. E o ρ já prevê que o Seidel é mais rápido: ρ(M_S) ≈ ρ(M_J)² (0,26 ≈ 0,25, comum em matrizes simétricas como essa).

### b) Gauss-Jacobi

In [19]:
x_j, k_j, info_j, hist_j = jacobi(A, b, x0, 1e-6, 1e-8, 1000)
print("\nb) Gauss-Jacobi")
imprimir_tabela(["k", "V1", "V2", "V3", "V4", "V5", "dr", "||b-Ax||"], hist_j)
print("V =", x_j, "| iterações =", k_j, "| info =", info_j)


b) Gauss-Jacobi
           k |           V1 |           V2 |           V3 |           V4 |           V5 |           dr |     ||b-Ax||
------------------------------------------------------------------------------------------------------------------------
           1 |          1.2 |     0.555556 |            0 |     0.555556 |         1.25 |            1 |      4.67222
           2 |      1.31111 |     0.883951 |     0.467222 |     0.895062 |      1.38889 |       0.3364 |       1.5858
           3 |      1.42351 |      1.05019 |     0.625802 |      1.06624 |      1.53217 |     0.111724 |     0.930523
           4 |      1.47262 |      1.12943 |     0.718855 |      1.15179 |      1.59479 |    0.0583478 |     0.441302
           5 |      1.49777 |      1.17053 |     0.762985 |      1.19519 |       1.6278 |    0.0271103 |      0.22716
           6 |       1.5104 |      1.19074 |     0.785701 |       1.2169 |      1.64417 |    0.0138161 |     0.112856
           7 |      1.51672 |      1

### c) Gauss-Seidel

In [20]:
x_s, k_s, info_s, hist_s = seidel(A, b, x0, 1e-6, 1e-8, 1000)
print("\nc) Gauss-Seidel")
imprimir_tabela(["k", "V1", "V2", "V3", "V4", "V5", "dr", "||b-Ax||"], hist_s)
print("V =", x_s, "| iterações =", k_s, "| info =", info_s)


c) Gauss-Seidel
           k |           V1 |           V2 |           V3 |           V4 |           V5 |           dr |     ||b-Ax||
------------------------------------------------------------------------------------------------------------------------
           1 |          1.2 |     0.822222 |     0.284444 |     0.710123 |      1.46309 |            1 |      2.92617
           2 |      1.39289 |       1.0072 |     0.629062 |      1.13239 |      1.61173 |     0.261995 |       1.1115
           3 |      1.46435 |      1.14658 |     0.763401 |      1.21076 |      1.64812 |    0.0845689 |     0.413098
           4 |      1.50566 |      1.19432 |     0.796393 |      1.23148 |      1.65742 |    0.0288044 |     0.128474
           5 |       1.5185 |      1.20681 |      0.80525 |      1.23691 |      1.65988 |   0.00773994 |    0.0338352
           6 |      1.52189 |      1.21013 |     0.807584 |      1.23834 |      1.66053 |   0.00203761 |   0.00897956
           7 |      1.52278 |      1

### d) critério de parada

já tá dentro das funções (toler_rel = 1e-6 e toler_res = 1e-8, simultâneos). As colunas dr e ||b-Ax|| das tabelas mostram os dois em cada iteração. o resíduo (1e-8) é o critério que manda (Jacobi para na it. 30 e Seidel na 17): o dr chega em 1e-6 antes. Por isso o "simultaneamente" importa: parar só pelo dr (Jacobi na it. 20, Seidel na it. 12) deixaria resíduo de ~1e-6 a 1e-5, bem acima de 1e-8.

### e) comparação

In [21]:
x_ref = np.linalg.solve(A, b) # referência direta (LU do numpy)
print("\ne) comparação")
print("referência (np.linalg.solve) =", x_ref)
print(f"Jacobi : {k_j} iterações | erro vs ref = {np.max(np.abs(x_j - x_ref)):.3e} | resíduo = {np.max(np.abs(b - A @ x_j)):.3e}")
print(f"Seidel : {k_s} iterações | erro vs ref = {np.max(np.abs(x_s - x_ref)):.3e} | resíduo = {np.max(np.abs(b - A @ x_s)):.3e}")
print("diferença Jacobi - Seidel =", np.max(np.abs(x_j - x_s)))


e) comparação
referência (np.linalg.solve) = [1.52310718 1.21132416 0.80842347 1.23885611 1.66076696]
Jacobi : 30 iterações | erro vs ref = 1.514e-09 | resíduo = 7.533e-09
Seidel : 17 iterações | erro vs ref = 5.378e-10 | resíduo = 3.956e-09
diferença Jacobi - Seidel = 1.2328043030862545e-09


**Resposta da letra e:** as duas soluções coincidem com a direta até ~1e-9: V ≈ (1,523 ; 1,211 ; 0,808 ; 1,239 ; 1,661). O Seidel chega lá em 17 iterações contra 30 do Jacobi (~43% a menos, na linha do que o ρ previa), usando só 1 vetor. O Jacobi só ganha se der pra paralelizar as n contas de cada iteração.

**Interpretação:** as tensões nodais são todas positivas e na mesma ordem de grandeza. V5 é a maior (b5 = 10 e a menor diagonal, 8) e V3 a menor (b3 = 0: o nó 3 n tem fonte, só recebe tensão dos vizinhos).

## Exercício 3: Tubulações em paralelo (Newton)
`F(q1, q2, h) = 0`, `x(0) = (0,060 ; 0,060 ; 5,0)`. Parada: `||Δx||∞ / max(||x(k+1)||∞, 1) < 1e-6` **ou** `||F(x(k+1))||∞ < 1e-8`.

### a) Jacobiana analítica
Derivadas feitas à mão e conferidas com o sympy.

In [22]:
# F e J numéricas. x = [q1, q2, h]
def F(x):
    q1, q2, h = x
    return np.array([q1 + q2 - 0.120,
                     h - 1200 * q1**2,
                     h - 2500 * q2**2])

# letra a) jacobiana analítica: J[i][j] = dFi/dxj, com x = (q1, q2, h)
#   F1 = q1 + q2 - 0,120      -> dF1/dq1 = 1        dF1/dq2 = 1        dF1/dh = 0
#   F2 = h - 1200 q1²         -> dF2/dq1 = -2400 q1 dF2/dq2 = 0        dF2/dh = 1
#   F3 = h - 2500 q2²         -> dF3/dq1 = 0        dF3/dq2 = -5000 q2 dF3/dh = 1
def J(x):
    q1, q2, h = x
    return np.array([[1.0,        1.0,         0.0],
                     [-2400 * q1, 0.0,         1.0],
                     [0.0,        -5000 * q2,  1.0]])

# confere a jacobiana da mão com a do sympy num ponto qualquer (se der ~0 a mão tá certa)
sq1, sq2, sh = sp.symbols('q1 q2 h')
F_sym = sp.Matrix([sq1 + sq2 - sp.Rational(12, 100), sh - 1200*sq1**2, sh - 2500*sq2**2])
J_sym = sp.lambdify([sq1, sq2, sh], F_sym.jacobian([sq1, sq2, sh]), "numpy")
ponto = [0.07, 0.05, 6.0]
print("a) J analítica no ponto", ponto, "=\n", J(ponto))
print("   |J analítica - J sympy| =", np.max(np.abs(J(ponto) - np.array(J_sym(*ponto)))))

a) J analítica no ponto [0.07, 0.05, 6.0] =
 [[   1.    1.    0.]
 [-168.    0.    1.]
 [   0. -250.    1.]]
   |J analítica - J sympy| = 0.0


### b) e c) Newton, resolvendo `J(xk) Δx = -F(xk)` em cada iteração

In [23]:
x0 = [0.060, 0.060, 5.0]
x, k, info, historico = newton(F, J, x0, 1e-6, 1e-8, 100)

print("\nb) e c) Newton, x0 =", x0)
imprimir_tabela(["k", "q1", "q2", "h", "Δq1", "Δq2", "Δh", "dr", "||F||"], historico)
print("x =", x, "| iterações =", k, "| info =", info)

# verificação: as duas perdas de carga têm que ser iguais a h, e as vazões somam 0,120
print("\nverificação: q1 + q2 =", x[0] + x[1], "| K1 q1² =", 1200 * x[0]**2, "| K2 q2² =", 2500 * x[1]**2, "| h =", x[2])
print("fração da vazão: tubo 1 =", x[0] / 0.120, "| tubo 2 =", x[1] / 0.120)
print("x (np.linalg.solve em cada iteração, só pra conferir) :", end=" ")
xc = np.array(x0, dtype=float)
for _ in range(10):
    xc = xc + np.linalg.solve(J(xc), -F(xc))
print(xc)

# guarda pro quadro final
res_ex3 = (k, np.max(np.abs(F(x))))


b) e c) Newton, x0 = [0.06, 0.06, 5.0]
           k |           q1 |           q2 |            h |          Δq1 |          Δq2 |           Δh |           dr |        ||F||
---------------------------------------------------------------------------------------------------------------------------------------
           0 |         0.06 |         0.06 |            5 |            - |            - |            - |            - |            4
           1 |    0.0705405 |    0.0494595 |      5.83784 |    0.0105405 |   -0.0105405 |     0.837838 |     0.143519 |     0.277757
           2 |    0.0708872 |    0.0491128 |      6.02986 |  0.000346701 | -0.000346701 |     0.192019 |    0.0318447 |  0.000300505
           3 |    0.0708876 |    0.0491124 |      6.03007 |  3.75908e-07 | -3.75908e-07 |  0.000208195 |  3.45262e-05 |  3.53268e-10
x = [0.07088762 0.04911238 6.03006522] | iterações = 3 | info = 0

verificação: q1 + q2 = 0.12 | K1 q1² = 6.03006522243902 | K2 q2² = 6.03006522262272 | h = 6.

**Resposta da letra a:** a jacobiana tá acima, e o sympy confirma (diferença ~0).

**Resposta das letras b e c:** converge em 3 iterações, parando pelo critério ||F|| < 1e-8 (o do passo ainda tava em ~3e-5 na iteração 3).

- q1 ≈ 0,07089 m³/s, q2 ≈ 0,04911 m³/s e h ≈ 6,030 (perda de carga comum às duas tubulações; a unidade segue a de K, provavelmente m se K estiver em s²/m⁵).
- Nas tabelas dá pra ver a convergência quadrática: ||F|| vai 2,8e-1 -> 3,0e-4 -> 3,5e-10 (os dígitos corretos dobram a cada passo).
- F1 é linear, então já era 0 em x0 e continua 0 em todas as iterações.

**Interpretação:** a tubulação 1 (K = 1200, menos resistência) leva ≈ 59% da vazão e a 2 (K = 2500) ≈ 41%. Como a perda de carga h é a mesma nas duas (K1 q1² = K2 q2²), a razão das vazões é q1/q2 = sqrt(K2/K1) ≈ 1,443. As duas vazões deram positivas e somam 0,120, como tinha que ser.

## Exercício 4: Fluxo de potência de duas barras (Newton Modificado)
`F(V, δ) = 0` com `V1 = 1`, `X = 0,5`, `PL = 0,6`, `QL = 0,2`, para `x(0) = (1,0 ; 0,20)` e `x(0) = (0,45 ; 0,75)`. δ em radianos. O enunciado não dá o critério de parada, então usei o mesmo do exercício 3.

### a) Jacobiana analítica

In [24]:
V1 = 1.0   # pu
X = 0.5    # pu
PL = 0.6   # pu
QL = 0.2   # pu

# x = [V, δ]
def F(x):
    V, d = x
    return np.array([V1 * V / X * np.sin(d) - PL,
                     V1 * V / X * np.cos(d) - V**2 / X - QL])

# letra a) jacobiana analítica: J[i][j] = dFi/dxj, com x = (V, δ)
#   F1 = (V1 V / X) sin δ - PL           -> dF1/dV = (V1/X) sin δ           dF1/dδ =  (V1 V / X) cos δ
#   F2 = (V1 V / X) cos δ - V²/X - QL    -> dF2/dV = (V1/X) cos δ - 2V/X    dF2/dδ = -(V1 V / X) sin δ
def J(x):
    V, d = x
    return np.array([[V1 / X * np.sin(d),             V1 * V / X * np.cos(d)],
                     [V1 / X * np.cos(d) - 2 * V / X, -V1 * V / X * np.sin(d)]])

# confere a jacobiana da mão com a do sympy num ponto qualquer
sV, sd = sp.symbols('V d')
F_sym = sp.Matrix([V1 * sV / X * sp.sin(sd) - PL, V1 * sV / X * sp.cos(sd) - sV**2 / X - QL])
J_sym = sp.lambdify([sV, sd], F_sym.jacobian([sV, sd]), "numpy")
ponto = [0.8, 0.4]
print("a) J analítica no ponto", ponto, "=\n", J(ponto))
print("   |J analítica - J sympy| =", np.max(np.abs(J(ponto) - np.array(J_sym(*ponto)))))

a) J analítica no ponto [0.8, 0.4] =
 [[ 0.77883668  1.47369759]
 [-1.35787801 -0.62306935]]
   |J analítica - J sympy| = 0.0


### b) e c) Newton Modificado para as duas estimativas iniciais

In [25]:
# valor analítico das raízes pra conferir (com V1 = 1): V sin δ = PL X = a   e   V cos δ = V² + QL X
# elevando ao quadrado e somando: V² = a² + (V² + c)²  ->  u² + (2c - 1) u + (a² + c²) = 0, com u = V²
a = PL * X
c = QL * X
u = np.roots([1, 2 * c - 1, a**2 + c**2])
print("\nraízes analíticas: V =", np.sqrt(u), "| δ =", [np.arcsin(a / np.sqrt(ui)) for ui in u], "rad")

resultados = []
for x0 in ([1.0, 0.20], [0.45, 0.75]):
    print("\nb) Newton Modificado, x0 =", x0, "| det J(x0) =", np.linalg.det(J(x0)))
    x, k, info, historico = newton_modificado(F, J, x0, 1e-6, 1e-8, 500)
    imprimir_tabela(["k", "V", "δ (rad)", "||Δx||", "dr", "||F||"], historico)
    print(f"c) V = {x[0]:.6f} pu | δ = {x[1]:.6f} rad = {np.degrees(x[1]):.4f}° | iterações = {k} | info = {info}")
    resultados.append((x0, x, k))

print("\nd) comparação das duas raízes")
for x0, x, k in resultados:
    print(f"  x0 = {x0} -> V = {x[0]:.6f} pu, δ = {np.degrees(x[1]):.4f}°, {k} iterações")
print("  |V1 - V2| =", abs(resultados[0][1][0] - resultados[1][1][0]))


raízes analíticas: V = [0.80308715 0.39376519] | δ = [np.float64(0.3828422450931511), np.float64(0.8662035273051033)] rad

b) Newton Modificado, x0 = [1.0, 0.2] | det J(x0) = 3.840532622729933
           k |            V |      δ (rad) |       ||Δx|| |           dr |        ||F||
------------------------------------------------------------------------------------------
           0 |            1 |          0.2 |            - |            - |     0.239867
           1 |     0.856609 |     0.332458 |     0.143391 |     0.143391 |    0.0481515
           2 |     0.827806 |     0.359143 |     0.028803 |     0.028803 |    0.0205448
           3 |     0.815448 |     0.370881 |    0.0123581 |    0.0123581 |   0.00990271
           4 |     0.809473 |     0.376634 |   0.00597522 |   0.00597522 |    0.0050228
           5 |     0.806437 |     0.379578 |   0.00303563 |   0.00303563 |   0.00261036
           6 |     0.804858 |     0.381114 |   0.00157894 |   0.00157894 |   0.00137333
           

**Resposta da letra a:** a jacobiana tá acima e o sympy confirma (diferença ~0).

**Resposta das letras b e c:**

- x0 = (1,0 ; 0,20) -> V ≈ 0,8031 pu e δ ≈ 0,3828 rad ≈ 21,9°
- x0 = (0,45 ; 0,75) -> V ≈ 0,3938 pu e δ ≈ 0,8662 rad ≈ 49,6°
- as duas batem com as raízes analíticas (V² = 0,6450 e 0,1550), e cada uma precisou de 18 e 11 iterações.

**Resposta da letra d:** o sistema tem DUAS soluções (mesmo sistema, estimativas iniciais diferentes -> raízes diferentes), e cada chute cai na raiz da sua região. Em sistemas elétricos:

- a raiz de V ≈ 0,80 pu e δ ≈ 22° é a solução de alta tensão, a que faz sentido operar (barra de carga com tensão razoável).
- a raiz de V ≈ 0,39 pu e δ ≈ 50° é a solução de baixa tensão: matematicamente válida, mas tensão muito abaixo do aceitável (~0,95 a 1,05 pu) e fisicamente instável/inviável. Aparece quando o chute inicial é ruim (V baixo e δ alto).
- Então o x0 escolhe a raiz, e saber qual é a fisicamente útil exige conhecimento do problema, não só do método.

**Convergência:** como a J é congelada, a convergência é linear: 18 iterações pro 1º chute e 11 pro 2º, com ||Δx|| caindo por um fator ~constante (≈ 0,5 e ≈ 0,3 por passo), bem diferente do Newton normal, que dobraria os dígitos corretos.

**Detalhe:** det J tem sinal diferente nas duas raízes (≈ +1,57 na de alta tensão e ≈ -0,77 na de baixa). J é singular no ponto de máximo carregamento, que fica entre os dois ramos, e o sinal do det J separa o ramo estável do instável.

## Quadro comparativo dos métodos

In [26]:
# (A e b aqui são os do exercício 2; F é a do exercício 4)
def residuo(A, b, x):
    return np.max(np.abs(np.array(b, dtype=float) - np.array(A, dtype=float) @ x))

linhas = [
    ["LU",                          "Ex 1 (4x4)", "direto", res_lu],
    ["Cholesky",                    "Ex 1 (4x4)", "direto", res_ch],
    ["Gauss-Jacobi",                "Ex 2 (5x5)", k_j,      residuo(A, b, x_j)],
    ["Gauss-Seidel",                "Ex 2 (5x5)", k_s,      residuo(A, b, x_s)],
    ["Newton",                      "Ex 3 (3x3)", res_ex3[0], res_ex3[1]],
]
for x0_, x_, k_ in resultados:
    linhas.append([f"Newton Modificado x0={x0_}", "Ex 4 (2x2)", k_, np.max(np.abs(F(x_)))])

print(f"{'método':<36}{'problema':<14}{'iterações':>10}   ||resíduo||∞")
print("-" * 76)
for nome, prob, it, r in linhas:
    print(f"{nome:<36}{prob:<14}{str(it):>10}   {r:.2e}")

método                              problema       iterações   ||resíduo||∞
----------------------------------------------------------------------------
LU                                  Ex 1 (4x4)        direto   5.68e-14
Cholesky                            Ex 1 (4x4)        direto   5.68e-14
Gauss-Jacobi                        Ex 2 (5x5)            30   7.53e-09
Gauss-Seidel                        Ex 2 (5x5)            17   3.96e-09
Newton                              Ex 3 (3x3)             3   3.53e-10
Newton Modificado x0=[1.0, 0.2]     Ex 4 (2x2)            18   7.15e-07
Newton Modificado x0=[0.45, 0.75]   Ex 4 (2x2)            11   2.31e-07
